<a href="https://colab.research.google.com/github/Subhrajeet-01/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [19]:
!git clone https://github.com/Subhrajeet-01/flyrank-ml-internship-starter.git

fatal: destination path 'flyrank-ml-internship-starter' already exists and is not an empty directory.


In [20]:
# ============================================================
# ML-05 SETUP — Load FlyRank starter dataset
# ============================================================

!pip -q install datasets pandas scikit-learn

import pandas as pd
import numpy as np

from datasets import load_dataset

# Public anonymized starter dataset
ds = load_dataset("FlyRank/internship-starter", split="train")

df = ds.to_pandas()

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Dataset shape: (30000, 53)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'health_score', 'needs_indexing', 'is_quick_win', 'needs_ctr_fix', 'needs_engagement_fix', 'ai_opportunity', 'is_underperformer', 'is_declining', 'is_initial_refresh_candidate']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_pct,health_score,needs_indexing,is_quick_win,needs_ctr_fix,needs_engagement_fix,ai_opportunity,is_underperformer,is_declining,is_initial_refresh_candidate
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,-41.4,50,False,False,False,False,False,False,True,True
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,-57.7,40,False,True,False,False,False,False,True,True
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,-60.9,40,False,False,False,False,False,False,True,False
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,-13.8,60,False,False,False,True,False,False,False,True
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,-34.7,40,False,False,False,True,True,False,True,False


In [21]:
# Basic schema check

print("Rows:", len(df))
print("Columns:", len(df.columns))

display(
    pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2)
    }).sort_values("missing_pct", ascending=False)
)

Rows: 30000
Columns: 53


,dtype,missing,missing_pct
provider_used,object,21438,71.46
word_count,float64,7699,25.66
char_count,float64,7699,25.66
char_count_tier,object,7699,25.66
word_count_tier,object,7699,25.66
model_used,object,5733,19.11
trend_pct,float64,3388,11.29
competition_level,object,2610,8.70
cpc,float64,2468,8.23
search_volume,float64,2468,8.23


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

## 1. Build the feature vector

The feature vector represents information that is available before the decision moment and can be used to rank content for refresh review.

For this refresh/content-opportunity task, I use observable search, engagement, content-age, and content-structure signals.

The feature vector contains numeric signals such as impressions, clicks, sessions, CTR, average position, content age, freshness, and content length, together with selected categorical fields where available.

I exclude identifiers such as `content_id` and `client_id` from the predictive feature vector because they are join/grouping keys rather than meaningful measurements.

The feature vector is constructed with numeric missing-value imputation and categorical missing-value handling. No future-window measurement or label-derived product decision is intentionally included.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# 1. BUILD FEATURE VECTOR
# ============================================================

# Candidate numeric features that may exist in the starter dataset.
numeric_candidates = [
    "search_volume",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

# Candidate categorical features.
categorical_candidates = [
    "competition",
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier"
]

numeric_features = [c for c in numeric_candidates if c in df.columns]
categorical_features = [c for c in categorical_candidates if c in df.columns]

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

# Build numeric feature matrix.
X_numeric = df[numeric_features].copy()

# Numeric missing-value handling.
X_numeric = X_numeric.replace([np.inf, -np.inf], np.nan)

for col in numeric_features:
    X_numeric[col] = X_numeric[col].fillna(X_numeric[col].median())

# Build categorical feature matrix.
X_categorical = df[categorical_features].copy()

# Categorical missing-value handling.
for col in categorical_features:
    X_categorical[col] = X_categorical[col].astype("string").fillna("missing")

# One-hot encode categorical fields.
if categorical_features:
    X_categorical_encoded = pd.get_dummies(
        X_categorical,
        columns=categorical_features,
        dtype=int
    )
else:
    X_categorical_encoded = pd.DataFrame(index=df.index)

# Final feature vector.
X = pd.concat(
    [X_numeric, X_categorical_encoded],
    axis=1
)

print("\nFinal feature-vector shape:", X.shape)

display(X.head())

Numeric features:
['search_volume', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Categorical features:
['competition', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']

Final feature-vector shape: (30000, 153)


,search_volume,cpc,word_count,char_count,impressions_90d,clicks_90d,sessions_90d,ai_sessions_90d,days_with_impressions,days_with_sessions,...,word_count_tier_missing,impression_tier_excellent,impression_tier_good,impression_tier_low,impression_tier_moderate,position_tier_deep,position_tier_page_1,position_tier_page_3_5,position_tier_striking,position_tier_top_3
0,10.0,2.05,3221.0,20457.0,3803,29,17,0,88,13,...,0,0,1,0,0,0,0,0,1,0
1,90.0,0.05,2481.0,15562.0,15320,7,9,0,88,9,...,0,0,1,0,0,0,0,1,0,0
2,0.0,0.00,3515.0,23643.0,12581,11,11,0,88,11,...,0,0,1,0,0,0,0,1,0,0
3,10.0,0.00,2877.0,19116.0,11751,58,78,0,88,51,...,1,0,1,0,0,0,1,0,0,0
4,0.0,0.00,2803.0,17469.0,19140,24,145,0,88,33,...,0,0,1,0,0,0,0,1,0,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## 2. Feature notes (meaning, missing, categorical, available-when?)

The selected features represent observable information available before a refresh-review ranking decision.

- `impressions_90d`: search exposure during the observed 90-day window.
- `clicks_90d`: search clicks during the observed 90-day window.
- `sessions_90d`: observed sessions during the feature window.
- `ctr`: observed click-through rate.
- `avg_position`: observed average search position.
- `content_age_days`: age of the content at the observation point.
- `days_since_last_update`: elapsed time since the latest content update.
- `word_count` / `char_count`: content-size measurements.
- `search_volume` / `cpc`: available search-demand/context measurements.
- `engagement_rate` / `scroll_rate`: observed engagement measurements.
- categorical fields such as `content_type`, `main_intent`, and tier fields provide structured context.

Numeric missing values are filled with the median of the available data. Categorical missing values are represented as `missing`.

The identifiers `content_id` and `client_id` are not used as predictive features.

A feature is considered valid only when it is observable before the decision point. A label-derived field, future-window measurement, or product decision flag is not treated as a normal feature.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# 2. FEATURE NOTES / AVAILABILITY CHECK
# ============================================================

feature_notes = []

for col in numeric_features + categorical_features:
    feature_notes.append({
        "feature": col,
        "type": "numeric" if col in numeric_features else "categorical",
        "missing_before_fill": int(df[col].isna().sum()),
        "missing_pct": round(df[col].isna().mean() * 100, 2),
        "available_in_dataset": True
    })

feature_notes_df = pd.DataFrame(feature_notes)

display(feature_notes_df)

,feature,type,missing_before_fill,missing_pct,available_in_dataset
0,search_volume,numeric,2468,8.23,True
1,cpc,numeric,2468,8.23,True
2,word_count,numeric,7699,25.66,True
3,char_count,numeric,7699,25.66,True
4,impressions_90d,numeric,0,0.00,True
5,clicks_90d,numeric,0,0.00,True
6,sessions_90d,numeric,0,0.00,True
7,ai_sessions_90d,numeric,0,0.00,True
8,days_with_impressions,numeric,0,0.00,True
9,days_with_sessions,numeric,0,0.00,True


In [24]:
# Check which obvious identifiers / decision fields exist.
identifier_candidates = [
    "content_id",
    "client_id",
    "url_hash_id",
    "keyword_hash_id"
]

decision_candidates = [
    "health_score",
    "priority_score",
    "action_type",
    "needs_ctr_fix",
    "is_quick_win",
    "refresh_tier",
    "is_declining_label"
]

print("Identifiers found:")
print([c for c in identifier_candidates if c in df.columns])

print("\nPotential decision/label-derived fields found:")
print([c for c in decision_candidates if c in df.columns])

Identifiers found:
['content_id', 'client_id']

Potential decision/label-derived fields found:
['health_score', 'needs_ctr_fix', 'is_quick_win']


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

## 3. The leakage hunt

I tested the feature set for information leakage.

The main leakage risk is using a field that is derived from the target or from a product decision that already encodes the target. Such a feature can make model performance appear much stronger than it really is.

I therefore compare a model using only observable features against a deliberately contaminated model containing a label-derived/product-decision field.

The contaminated model is used only as a leakage demonstration. It is not retained in the final feature vector.

I also check for future-window fields and fields that directly represent a product decision.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# 3. LEAKAGE HUNT
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# ------------------------------------------------------------
# Find the starter label.
# ------------------------------------------------------------

if "is_declining_label" in df.columns:
    y = df["is_declining_label"].astype(int)

elif "trend_direction" in df.columns:
    y = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

else:
    raise ValueError(
        "Could not find the expected decline label or trend_direction."
    )

print("Target distribution:")
print(y.value_counts())
print("\nTarget rate:", round(y.mean(), 4))

Target distribution:
trend_direction
1    16262
0    13738
Name: count, dtype: int64

Target rate: 0.5421


In [26]:
# ============================================================
# HONEST MODEL — observable features only
# ============================================================

# Use only numeric features for a simple leakage experiment.
honest_features = [
    c for c in numeric_features
    if c not in [
        "trend_pct",
        "trend_direction",
        "is_declining_label"
    ]
]

X_honest = df[honest_features].copy()

# Replace invalid values.
X_honest = X_honest.replace([np.inf, -np.inf], np.nan)

X_train, X_test, y_train, y_test = train_test_split(
    X_honest,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

honest_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

honest_model.fit(X_train, y_train)

honest_prob = honest_model.predict_proba(X_test)[:, 1]

honest_auc = roc_auc_score(y_test, honest_prob)
honest_ap = average_precision_score(y_test, honest_prob)

print("HONEST MODEL")
print("ROC AUC:", round(honest_auc, 4))
print("Average Precision:", round(honest_ap, 4))

HONEST MODEL
ROC AUC: 0.6781
Average Precision: 0.6951


In [27]:
# ============================================================
# DELIBERATELY LEAKY MODEL
# ============================================================

# Look for a field that directly represents or strongly encodes
# the target/product decision.

leak_candidates = [
    "is_declining_label",
    "trend_direction",
    "health_score",
    "priority_score",
    "action_type",
    "needs_ctr_fix",
    "is_quick_win"
]

available_leaks = [
    c for c in leak_candidates
    if c in df.columns
]

print("Potential leakage fields available:")
print(available_leaks)

Potential leakage fields available:
['trend_direction', 'health_score', 'needs_ctr_fix', 'is_quick_win']


In [28]:
# ============================================================
# TEST THE FIRST AVAILABLE LEAKAGE FIELD
# ============================================================

leak_col = None

# Prefer direct target leakage for demonstration.
for candidate in [
    "is_declining_label",
    "trend_direction",
    "health_score",
    "priority_score",
    "needs_ctr_fix",
    "is_quick_win",
    "action_type"
]:
    if candidate in df.columns:
        leak_col = candidate
        break

print("Selected leakage demonstration field:", leak_col)

if leak_col is None:
    print(
        "No obvious leakage field was found. "
        "The honest feature audit above is still valid."
    )
else:
    leak_df = df[[leak_col]].copy()

    # Convert categorical leakage fields into numeric codes.
    if not pd.api.types.is_numeric_dtype(leak_df[leak_col]):
        leak_df[leak_col] = (
            leak_df[leak_col]
            .astype("category")
            .cat.codes
        )

    leak_df[leak_col] = leak_df[leak_col].replace(
        [-1, np.inf, -np.inf],
        np.nan
    )

    X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
        leak_df,
        y,
        test_size=0.25,
        random_state=42,
        stratify=y
    )

    leaky_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])

    leaky_model.fit(X_train_l, y_train_l)

    leaky_prob = leaky_model.predict_proba(X_test_l)[:, 1]

    leaky_auc = roc_auc_score(y_test_l, leaky_prob)
    leaky_ap = average_precision_score(y_test_l, leaky_prob)

    print("\nLEAKY MODEL")
    print("ROC AUC:", round(leaky_auc, 4))
    print("Average Precision:", round(leaky_ap, 4))

    print("\nComparison")
    print("-----------------------------")
    print("Honest ROC AUC:", round(honest_auc, 4))
    print("Leaky  ROC AUC:", round(leaky_auc, 4))
    print("-----------------------------")

Selected leakage demonstration field: trend_direction

LEAKY MODEL
ROC AUC: 1.0
Average Precision: 1.0

Comparison
-----------------------------
Honest ROC AUC: 0.6781
Leaky  ROC AUC: 1.0
-----------------------------


In [29]:
# ============================================================
# LEAKAGE DECISION
# ============================================================

print("LEAKAGE AUDIT SUMMARY")
print("======================")

print("Honest features:", honest_features)

print("\nExcluded leakage candidates:")
print(available_leaks)

print("\nConclusion:")
print(
    "Fields derived from the target or product decisions are "
    "excluded from the final feature vector because they can "
    "leak the answer into the model."
)

LEAKAGE AUDIT SUMMARY
Honest features: ['search_volume', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Excluded leakage candidates:
['trend_direction', 'health_score', 'needs_ctr_fix', 'is_quick_win']

Conclusion:
Fields derived from the target or product decisions are excluded from the final feature vector because they can leak the answer into the model.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## 4. What I excluded and why

I excluded the following fields from the final predictive feature vector:

- `content_id` and `client_id`: identifiers used for grouping or joining, not predictive measurements.
- `trend_direction`: directly related to the decline label and therefore unsuitable as a predictive feature.
- `is_declining_label`: the target itself; using it as a feature would directly reveal the answer.
- `health_score`: a product-derived decision signal that can encode the underlying decline/priority logic.
- `priority_score`: a product decision score and therefore a circular feature if used to rediscover the same decision.
- `action_type`: a product decision rather than an independent observable signal.
- `needs_ctr_fix` and similar decision flags: product-generated outputs that may encode the target or action.
- Any future-window metric: unavailable at prediction time and therefore a source of temporal leakage.
- Raw client names, URLs, queries, titles, or other identifying fields: not appropriate for the public-safe internship analysis.

The final feature vector therefore contains observable, pre-decision measurements and excludes identifiers, target-derived fields, product decisions, future information, and sensitive/raw-origin information.

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# 4. EXCLUDED FIELDS AUDIT
# ============================================================

excluded_reasons = {
    "content_id": "Identifier; useful for grouping but not a predictive measurement.",
    "client_id": "Identifier; useful for grouping/validation but not a predictive measurement.",
    "trend_direction": "Target-related current-state signal; can leak the decline label.",
    "is_declining_label": "Direct target; using it as a feature reveals the answer.",
    "health_score": "Product-derived decision signal; may encode the same decision.",
    "priority_score": "Product-derived ranking signal; creates circularity if rediscovering priority.",
    "action_type": "Product decision output, not an independent observable feature.",
    "needs_ctr_fix": "Product decision flag that may encode the desired action.",
    "is_quick_win": "Product decision flag and potential target leakage.",
    "refresh_tier": "Product/decision-derived categorization rather than an independent signal.",
    "url_hash_id": "Join/grouping identifier; not used as a predictive measurement.",
    "keyword_hash_id": "Join/grouping identifier; not used as a predictive measurement."
}

excluded_rows = []

for field, reason in excluded_reasons.items():
    if field in df.columns:
        excluded_rows.append({
            "field": field,
            "reason": reason
        })

excluded_df = pd.DataFrame(excluded_rows)

display(excluded_df)

,field,reason
0,content_id,Identifier; useful for grouping but not a pred...
1,client_id,Identifier; useful for grouping/validation but...
2,trend_direction,Target-related current-state signal; can leak ...
3,health_score,Product-derived decision signal; may encode th...
4,needs_ctr_fix,Product decision flag that may encode the desi...
5,is_quick_win,Product decision flag and potential target lea...


In [31]:
# ============================================================
# FINAL FEATURE VECTOR CHECK
# ============================================================

final_feature_columns = [
    c for c in X.columns
    if c not in excluded_reasons
]

print("Final feature count:", len(final_feature_columns))

print("\nFinal feature columns:")
print(final_feature_columns)

# Confirm excluded fields are not present.
remaining_excluded = [
    c for c in final_feature_columns
    if c in excluded_reasons
]

print("\nExcluded fields accidentally remaining:")
print(remaining_excluded)

assert len(remaining_excluded) == 0

print("\nFeature-vector leakage check: PASSED")

Final feature count: 153

Final feature columns:
['search_volume', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'competition_0.0', 'competition_0.01', 'competition_0.02', 'competition_0.03', 'competition_0.04', 'competition_0.05', 'competition_0.06', 'competition_0.07', 'competition_0.08', 'competition_0.09', 'competition_0.1', 'competition_0.11', 'competition_0.12', 'competition_0.13', 'competition_0.14', 'competition_0.15', 'competition_0.16', 'competition_0.17', 'competition_0.18', 'competition_0.19', 'competition_0.2', 'competition_0.21', 'competition_0.22', 'competition_0.23', 'competition_0.24', 'competition_0.25', 'competition_0.26', 'competition_0.27', 'competition_0.28', 'competition_0.29', 'competition_0.3', 'competition_0.31', 'competition_0.32', 'compe

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.